<a href="https://colab.research.google.com/github/fbeilstein/bioinformatics/blob/master/practice_02_evolutionary_divergence.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [5]:
import os
import time
from IPython.display import clear_output

# Ensure Biopython is installed before importing
os.system("pip install -q biopython")
clear_output()

from Bio import Entrez, SeqIO
print("Biopython installed and imported successfully.")

# Always provide an email when using NCBI Entrez
Entrez.email = "student@example.com"

# Accession numbers for Hemoglobin alpha (HBA) mRNA across 5 vertebrates
globin_accessions = {
    "Human": "NM_000558",
    "Mouse": "NM_008218",
    "Chicken": "NM_205104",
    "Frog": "NM_001096338",
    "Zebrafish": "NM_131144"
}

sequences = {}

print("Fetching sequences from NCBI...")
for species, acc in globin_accessions.items():
    try:
        # Fetch the FASTA record
        handle = Entrez.efetch(db="nucleotide", id=acc, rettype="fasta", retmode="text")
        record = SeqIO.read(handle, "fasta")
        sequences[species] = record.seq
        print(f"Downloaded {species} ({acc}): {len(record.seq)} bp")
        time.sleep(1) # Prevent 502 errors from rate limiting
    except Exception as e:
        print(f"Failed to fetch {species}: {e}")

Biopython installed and imported successfully.
Fetching sequences from NCBI...
Downloaded Human (NM_000558): 577 bp
Downloaded Mouse (NM_008218): 569 bp
Downloaded Chicken (NM_205104): 2280 bp
Downloaded Frog (NM_001096338): 849 bp
Downloaded Zebrafish (NM_131144): 1020 bp


### Dynamic Programming: The Needleman-Wunsch Algorithm

To determine the evolutionary distance between two sequences, we must align them. Finding the optimal global alignment is computationally expensive. If we align two sequences of length $N$, a brute-force approach would take exponential time.

Instead, we use the **Needleman-Wunsch algorithm**, which relies on dynamic programming. It builds a scoring matrix to evaluate all possible alignments, requiring $O(N^2)$ time and $O(N^2)$ memory space.

The alignment is governed by a scoring system reflecting biological realities:
*   **Match:** Positive score. The nucleotide was conserved.
*   **Mismatch:** Negative penalty. A point mutation occurred. (Note: biological models often penalize transversions (A↔C) more heavily than transitions (A↔G) because they are chemically less likely).
*   **Gap Open:** Large negative penalty. Represents a replication slippage event causing an insertion or deletion (indel).
*   **Gap Extend:** Smaller negative penalty. It is biologically more likely for one large indel of 5 bases to occur than 5 separate 1-base indels. This two-tier penalty is called an **affine gap penalty**.

**Configuring the Pairwise Aligner**

This block configures the Biopython C-accelerated aligner to perform the Needleman-Wunsch algorithm using specific affine gap penalties.

In [6]:
from Bio import Align

# Initialize the aligner
aligner = Align.PairwiseAligner()

# Force global alignment (Needleman-Wunsch)
aligner.mode = 'global'

# Set biological scoring parameters
aligner.match_score = 5
aligner.mismatch_score = -4

# Affine gap penalties
aligner.open_gap_score = -10
aligner.extend_gap_score = -1

print("Aligner configured with Needleman-Wunsch global alignment.")
print(aligner)

Aligner configured with Needleman-Wunsch global alignment.
Pairwise sequence aligner with parameters
  wildcard: None
  match_score: 5.000000
  mismatch_score: -4.000000
  open_internal_insertion_score: -10.000000
  extend_internal_insertion_score: -1.000000
  open_left_insertion_score: -10.000000
  extend_left_insertion_score: -1.000000
  open_right_insertion_score: -10.000000
  extend_right_insertion_score: -1.000000
  open_internal_deletion_score: -10.000000
  extend_internal_deletion_score: -1.000000
  open_left_deletion_score: -10.000000
  extend_left_deletion_score: -1.000000
  open_right_deletion_score: -10.000000
  extend_right_deletion_score: -1.000000
  mode: global



**Executing the Alignments and Calculating Divergence**

This block runs the pairwise alignments, isolating the human sequence and comparing it against the other vertebrates to demonstrate the molecular clock (the accumulation of mutations over evolutionary time).

In [7]:
# Isolate the human sequence as the reference
human_seq = sequences["Human"]

print("### Evolutionary Divergence from Human $\\alpha$-globin ###\n")

for species in ["Mouse", "Chicken", "Frog", "Zebrafish"]:
    target_seq = sequences[species]

    # Perform the alignment
    alignments = aligner.align(human_seq, target_seq)
    best_alignment = alignments[0]

    # Calculate a simplified identity percentage
    matches = str(best_alignment).count("|")
    length = max(len(human_seq), len(target_seq))
    identity = (matches / length) * 100

    print(f"Human vs {species}:")
    print(f"Alignment Score: {best_alignment.score}")
    print(f"Sequence Identity: {identity:.2f}%")

    # Print the first 100 bases of the alignment to visualize mutations and indels
    print(str(best_alignment)[:150])
    print("-" * 60)

### Evolutionary Divergence from Human $\alpha$-globin ###

Human vs Mouse:
Alignment Score: 1716.0
Sequence Identity: 78.86%
target            0 -ACTCTTCTGGTCCCCACAGACTCAGAGAGAACCCACCATGGTGCTGTCTCCTGCCGACA
                  0 -||.||||||.|.|..||||||||||..||||---|||||||||||.||
------------------------------------------------------------
Human vs Chicken:
Alignment Score: -660.0
Sequence Identity: 20.92%
target            0 -AC--------------------------------TC---TTCTGG---------TCCCC
                  0 -||--------------------------------||---|||.||---
------------------------------------------------------------
Human vs Frog:
Alignment Score: 149.0
Sequence Identity: 42.76%
target            0 ACTCTT-------CTGGTCCCCACAGACTCAGAGAG-----AACCCACCATGGTGCTGTC
                  0 .|..||-------.|..|.||..|||---|.||.||-----|...||..
------------------------------------------------------------
Human vs Zebrafish:
Alignment Score: 170.0
Sequence Identity: 38.82%
target            0 ACTCTTCTG--GTCCCC

### Analyzing the Output

Look at the textual alignments printed above.
*   The `|` symbol indicates a conserved nucleotide (no mutation).
*   A blank space indicates a **point mutation** (substitution).
*   A `-` symbol indicates a **gap** inserted by the algorithm to account for an **indel** (insertion/deletion) event caused by replication slippage.

Notice how the **Sequence Identity** correlates with established evolutionary timelines. The Mouse sequence has a higher identity to the Human sequence than the Zebrafish. This constant rate of mutation accumulation over millions of years serves as our **molecular clock**.